# One check

In [ ]:
import json
import time

import pandas as pd

from recon import activity, check, db, gap, identity, intent, jobs, observe, processing, queue, storage
from recon.config import settings
from recon.execution import SepexClient

pd.set_option("display.max_colwidth", 42)

print(f"database  {settings.postgres_host}:{settings.postgres_port}/{settings.postgres_db}")
print(f"storage   {settings.twod_fim_data_root_prefix} via {settings.aws_endpoint_url}")
print(f"sepex     {settings.sepex_url}")

## 1. Load Database and Author Intent (Only Needed for First Time)

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, "../scripts")
import author_intent
import aoi_config
import seed
import stage_source_data

stage_source_data.stage(Path("../testdata/lulc.tif"), "e2e/lulc.tif")
stage_source_data.stage(Path("../testdata/lulc_lookup.json"), "e2e/lulc_lookup.json")
e2e = aoi_config.load("../testdata/e2e.aoi_config.json")
seed.seed_water_bodies("lake", e2e)
seed.seed_network(e2e)
author_intent.author(e2e)

with db.connect() as conn:
    conn.execute("TRUNCATE materialized_models, materialized_nd_runs, "
                 "materialized_kwse_runs, reach_processing, reach_activity")

display(pd.DataFrame(db.table_counts()))


## 2. Database is the Queue

In [ ]:
due = pd.DataFrame(queue.due_reaches())
print(f"{len(due)} reaches due")
due.head(10)

## 3. Intent, and the address it implies

In [ ]:
rows = db.query("""
    SELECT rn.reach_id, rn.is_terminal FROM reach_network rn
    JOIN reach_network ds ON ds.reach_id = rn.reach_to_id
    WHERE ds.is_terminal LIMIT 1""")
upstream_id = rows[0]["reach_id"]
terminal_id = db.one("SELECT reach_to_id AS r FROM reach_network WHERE reach_id=%s", (upstream_id,))["r"]
print(f"terminal reach:     {terminal_id}")
print(f"non-terminal above: {upstream_id}\n")

wanted = intent.effective(terminal_id)
identity_obj, identity_hash = identity.model_identity(wanted)
print("identity object the job will build:")
print(json.dumps(identity_obj, indent=2))
print(f"\npredicted identity hash: {identity_hash}")
print(f"predicted address:       {storage.model_base_path(terminal_id)}/{identity_hash}_<domain>/")

## 4. Check - Part 1 Observe

In [ ]:
print("observe:", observe.observe_reach(terminal_id))
print("proof rows:", db.query("SELECT * FROM materialized_models"))

## 5. Check - Part 2 Gap

In [ ]:
for rid, label in ((terminal_id, "terminal"), (upstream_id, "non-terminal")):
    snap = check.load_snapshot(rid)
    print(f"{label} {rid}:")
    print(f"   snapshot: model_ok={snap.model_ok} ds_model_ok={snap.ds_model_ok} ds_nd_ok={snap.ds_nd_ok}")
    print(f"   decision: {gap.calculate(snap)}\n")

## 6. Check - Part 3 Act

In [ ]:
runner = SepexClient(base_url=settings.sepex_url)

print("terminal:    ", check.run_check(terminal_id, runner))
print("check again: ", check.run_check(terminal_id, runner), "   <- no resubmit")
print("non-terminal:", check.run_check(upstream_id, runner))
print()
pd.DataFrame(processing.in_flight())

## 7. Poll Jobs Separately

In [ ]:
deadline = time.time() + 900
while time.time() < deadline:
    outcomes = jobs.poll_in_flight(runner)
    if not outcomes:
        print("nothing in flight")
        break
    print(f"{time.strftime('%H:%M:%S')}  {outcomes[0]['status']:<10} {outcomes[0]['action']}")
    if outcomes[0]["status"] in ("succeeded", "failed"):
        break
    time.sleep(15)

## 8. The Next Check (via Observe) will Notice Desired State is Materialized now

In [ ]:
print(check.run_check(terminal_id, runner))
print()
row = db.one("SELECT * FROM materialized_models WHERE reach_id=%s", (terminal_id,))
print("proof row: ", row)
print(f"\npredicted {identity_hash} == adopted {row['identity_hash']}:", identity_hash == row["identity_hash"])